# Iceberg 1.12: Hilbert-curve clustering

**New in Iceberg 1.12.0 for Spark 4.1:** `rewrite_data_files` can cluster data along a **Hilbert curve**
(`sort_order => 'hilbert(x,y)'`), next to the older Z-order. Clustering matters when queries filter on *several* columns:
Iceberg prunes whole files using each column's min/max statistics, but only if the files do not all span the full range.

In this notebook you will
1. load the same 2-D point data into three tables as many small files (the worst case for pruning),
2. compact one table the normal way, one with Z-order and one with Hilbert,
3. **measure** how many files a spatial window query still has to open, and
4. check that Trino sees the same layout (the tables are plain format-v2 Iceberg tables).

Runs on Docker Compose (classic Spark) and on Kubernetes (Spark Connect) without changes.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))     # lab_utils.py lives one folder up
from lab_utils import get_spark, trino, CATALOG

spark = get_spark("iceberg-hilbert")
print("Spark", spark.version)
NS = f"{CATALOG}.iceberg12"
_ = spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {NS}")

## 1. Source data: 240,000 random points in 12 batches
Each batch will become one insert, so every table ends up with many small files, each covering the whole 0..1000 x 0..1000 square.

In [ ]:
from pyspark.sql import functions as F

N_BATCHES, ROWS_PER_BATCH = 12, 20_000
spark.sql(f"DROP TABLE IF EXISTS {NS}.points_src")
(spark.range(N_BATCHES * ROWS_PER_BATCH)
      .withColumn("batch", (F.col("id") / ROWS_PER_BATCH).cast("int"))
      .withColumn("x", F.rand(7) * 1000)
      .withColumn("y", F.rand(11) * 1000)
      .withColumn("payload", F.concat(F.lit("row-"), F.col("id").cast("string")))
      .writeTo(f"{NS}.points_src").using("iceberg").create())
spark.sql(f"SELECT count(*) AS points, min(x), max(x), min(y), max(y) FROM {NS}.points_src").show()

In [ ]:
VARIANTS = ["points_binpack", "points_zorder", "points_hilbert"]
for name in VARIANTS:
    spark.sql(f"DROP TABLE IF EXISTS {NS}.{name}")
    spark.sql(f"""CREATE TABLE {NS}.{name} (id BIGINT, x DOUBLE, y DOUBLE, payload STRING) USING iceberg
                  TBLPROPERTIES ('write.target-file-size-bytes' = '262144')""")
    for b in range(N_BATCHES):   # one small insert per batch -> many small files
        spark.sql(f"INSERT INTO {NS}.{name} SELECT id, x, y, payload FROM {NS}.points_src WHERE batch = {b}")
    n = spark.sql(f"SELECT count(*) FROM {NS}.{name}.files").collect()[0][0]
    print(f"{name:15s} {n} data files")

## 2. How well can a query skip files?
For a window `x in [x0,x1], y in [y0,y1]` a file can be skipped when its min/max bounds do not overlap the window.
The helper below reads those bounds from the table's `files` metadata table (the same statistics the planner uses) and
averages over 25 fixed windows of 100 x 100.

In [ ]:
import random
import pandas as pd

random.seed(1)
WINDOWS = [(x0, x0 + 100, y0, y0 + 100) for x0, y0 in ((random.uniform(0, 900), random.uniform(0, 900)) for _ in range(25))]

def pruning(table):
    files = spark.sql(f"""SELECT readable_metrics.x.lower_bound AS lx, readable_metrics.x.upper_bound AS ux,
                                  readable_metrics.y.lower_bound AS ly, readable_metrics.y.upper_bound AS uy
                           FROM {NS}.{table}.files""").toPandas()
    touched = [int(((files.ux >= x0) & (files.lx <= x1) & (files.uy >= y0) & (files.ly <= y1)).sum()) for x0, x1, y0, y1 in WINDOWS]
    return len(files), sum(touched) / len(touched)

rows = []
for name in VARIANTS:
    n, avg = pruning(name)
    rows.append({"table": name, "stage": "before compaction", "files": n, "avg files opened per window": round(avg, 1)})
pd.DataFrame(rows)

## 3. Compact: bin-pack vs Z-order vs Hilbert
All three rewrites use the same options (rewrite everything into ~256 KB files); only the **strategy** differs. The call is documented in the
[Spark procedures reference](https://iceberg.apache.org/docs/latest/spark-procedures/#rewrite_data_files).

In [ ]:
OPTIONS = "map('rewrite-all','true','target-file-size-bytes','262144','min-file-size-bytes','0')"
CALLS = {
    "points_binpack": "",                                                         # default strategy: bin-pack, no clustering
    "points_zorder":  ", strategy => 'sort', sort_order => 'zorder(x,y)'",
    "points_hilbert": ", strategy => 'sort', sort_order => 'hilbert(x,y)'",     # new in 1.12 (Spark 4.1)
}
for name, extra in CALLS.items():
    r = spark.sql(f"CALL {CATALOG}.system.rewrite_data_files(table => 'iceberg12.{name}'{extra}, options => {OPTIONS})").collect()[0]
    print(f"{name:15s} rewrote {r.rewritten_data_files_count} files into {r.added_data_files_count}")

In [ ]:
after = []
for name in VARIANTS:
    n, avg = pruning(name)
    after.append({"table": name, "stage": "after compaction", "files": n, "avg files opened per window": round(avg, 1)})
result = pd.DataFrame(rows + after)
print(result.pivot(index="table", columns="stage", values=["files", "avg files opened per window"]).to_string())
result[result.stage == "after compaction"].set_index("table")["avg files opened per window"].plot.bar(
    title="Files opened per 100x100 window (lower is better)", ylabel="files");

**Reading the result.** Bin-packing only merges small files, so each output file still spans the whole square and nearly every file must be opened.
Z-order and Hilbert both keep points that are close in 2-D close in the same file, so a window touches only a few files.
Exact numbers depend on the random data; the ordering (bin-pack far behind the two curves) is the point.

Z-order and Hilbert are both space-filling curves. The Hilbert curve avoids the long jumps of the Z curve, so it often (not always) clusters at least
as well; here it opens fewer files per window than Z-order. Whether the difference matters depends on your queries (small compact windows benefit most), so measure
on your own data, as done here.

## 4. Trino sees the same layout
These are ordinary format-v2 tables, so Trino reads them and exposes the same file statistics through its `$files` metadata tables.

In [ ]:
for name in VARIANTS:
    f = trino(f'SELECT count(*) AS files FROM iceberg.iceberg12."{name}$files"')
    c = trino(f"SELECT count(*) AS hits FROM iceberg.iceberg12.{name} WHERE x BETWEEN 100 AND 200 AND y BETWEEN 100 AND 200")
    print(f"{name:15s} Trino: {int(f.files[0])} files, {int(c.hits[0])} points in the window")

## Notes and limits
* `hilbert(...)` and `zorder(...)` cannot be combined with each other or with plain column sorts in one `sort_order`; Hilbert has no extra options.
* The feature was added for **Spark 4.1** in Iceberg 1.12 (release notes: "Spark 4.1: Add Hilbert-curve clustering strategy for `rewrite_data_files`").
* A rewrite reads and rewrites the data, so run it as scheduled maintenance, not per query. Use `where => '...'` to limit it to recent partitions.

In [ ]:
CLEANUP = False   # set True to drop the tables created here
if CLEANUP:
    for t in ["points_src"] + VARIANTS:
        spark.sql(f"DROP TABLE IF EXISTS {NS}.{t}")